# Module 11 — DETR Family (SOLUTIONS)

In [ ]:
import torch
import torch.nn.functional as F
from matcher import HungarianMatcher, box_cxcywh_to_xyxy, generalized_box_iou

def detr_loss(pred_logits, pred_boxes, tgt_classes, tgt_boxes, matcher,
             lambda_bbox=5.0, lambda_giou=2.0):
    """
    DETR bipartite loss.
    pred_logits: (B, N_q, num_classes+1)
    pred_boxes:  (B, N_q, 4) cxcywh
    tgt_classes: list of (N_gt,)
    tgt_boxes:   list of (N_gt, 4)
    """
    B, N_q, num_cls = pred_logits.shape
    indices = matcher(pred_logits, pred_boxes, tgt_classes, tgt_boxes)

    # Classification loss (all queries vs. no-object)
    no_obj_label = num_cls - 1  # last class = no object
    target_classes = torch.full((B, N_q), no_obj_label, dtype=torch.long,
                                 device=pred_logits.device)
    for b, (pred_idx, tgt_idx) in enumerate(indices):
        target_classes[b, pred_idx] = tgt_classes[b][tgt_idx]

    loss_ce = F.cross_entropy(pred_logits.flatten(0, 1),
                               target_classes.flatten(0, 1))

    # Box losses (matched pairs only)
    all_pred_boxes, all_tgt_boxes = [], []
    for b, (pred_idx, tgt_idx) in enumerate(indices):
        if len(pred_idx) == 0:
            continue
        all_pred_boxes.append(pred_boxes[b][pred_idx])
        all_tgt_boxes.append(tgt_boxes[b][tgt_idx])

    loss_bbox = loss_giou = torch.tensor(0.0, device=pred_logits.device)
    if all_pred_boxes:
        pb = torch.cat(all_pred_boxes)
        gb = torch.cat(all_tgt_boxes)
        loss_bbox = F.l1_loss(pb, gb)
        giou = generalized_box_iou(box_cxcywh_to_xyxy(pb), box_cxcywh_to_xyxy(gb))
        loss_giou = (1 - giou.diag()).mean()

    total = loss_ce + lambda_bbox * loss_bbox + lambda_giou * loss_giou
    return {'loss_ce': loss_ce.item(), 'loss_bbox': loss_bbox.item(),
            'loss_giou': loss_giou.item(), 'total': total.item()}


# Test
B, N_q, C = 2, 10, 81
pred_logits = torch.randn(B, N_q, C)
pred_boxes  = torch.rand(B, N_q, 4)
tgt_classes = [torch.tensor([1, 5]), torch.tensor([3])]
tgt_boxes   = [torch.rand(2, 4), torch.rand(1, 4)]
matcher = HungarianMatcher()
losses = detr_loss(pred_logits, pred_boxes, tgt_classes, tgt_boxes, matcher)
print('Losses:', {k: f'{v:.4f}' for k, v in losses.items()})